# Phase 4 — Signal Processing for the IGBT data

## What we are investigating

The square-gate aging files contain two distinct kinds of time-series measurement: low-speed steady-state records, and high-speed, oscilloscope-like transient waveform captures. Their sampling, physical meaning, and appropriate processing differ, so we must not concatenate or process them as if they were one uniform signal.

**Why this matters in PHM:** sampling metadata determines which time scales a signal can represent. Filtering can remove noise, but it can also erase switching edges and ringing that may be diagnostically meaningful. Normalization can help compare signal shapes, but can also hide amplitude changes. Windows and FFTs are meaningful only when the selected signal and analysis question justify them.

**Expected result:** inspect measured sample intervals, view an actual switching transient, see how a deliberately illustrative filter changes its waveform, and understand why per-record normalization and an FFT are not automatically degradation analysis.

**Scope:** we will not clean the source data, overwrite raw observations, extract PHM features, declare a noise-free signal, select a Health Indicator, or create RUL labels. All transformations below are temporary in-memory illustrations; the raw files remain untouched.

## 1. Two time scales in this experiment

Phase 2 found that steady-state records are typically about 0.2 seconds apart (roughly 5 records/second), with occasional larger gaps. Each transient record is an independent, short waveform capture with its **own** `dt` stored inside the MAT file. The actual transient captures inspected here each contain 125,000 samples, and the recorded `dt` varies between captures (4, 8, or 16 ns in this aging dataset). Therefore, compute the sample rate per capture:

$$ f_s = 1 / \Delta t, \qquad f_{Nyquist} = f_s / 2 $$

Do not infer a sample rate by combining capture timestamps, and do not concatenate separate captures before signal processing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import signal

project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

raw_root = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed" / "aging_steady_state_observations.csv"
aging_folders = sorted(raw_root.rglob("Aging Data"))
if len(aging_folders) != 1:
    raise FileNotFoundError(
        f"Expected one 'Aging Data' folder under {raw_root}; found {len(aging_folders)}."
    )
if not processed_path.is_file():
    raise FileNotFoundError("Run notebooks/02_data_loading.ipynb first to create the processed table.")

aging_folder = aging_folders[0]
steady = pd.read_csv(processed_path)
print("Steady-state table:", steady.shape)
print("Steady-state time column available:", "timeEpoch" in steady.columns)

mat_files = sorted(aging_folder.rglob("*.mat"))
transient_files = []
for mat_path in mat_files:
    data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if "measurement" in data and hasattr(data["measurement"], "transient"):
        records = np.atleast_1d(data["measurement"].transient)
        transient_files.append({"path": mat_path, "records": len(records)})

transient_inventory = pd.DataFrame(
    [{"source_file": item["path"].name, "transient_records": item["records"]}
     for item in transient_files]
)
transient_inventory

## 2. Sampling and time axes for a real transient

We select an observed capture from `Device2  1.mat` and verify its MATLAB fields before use. `dt` is the within-waveform sample spacing in seconds; `np.arange(N) * dt` constructs relative sample times. This is relative time since the capture started, **not** device aging time. We plot collector-emitter voltage because it is an actual field in this capture; the measurement README warns that steady-state data are improperly scaled, and absolute calibration of transient values still needs verification.

In [ ]:
device2_candidates = [path for path in mat_files if path.name.startswith("Device2 ")]
if len(device2_candidates) != 1:
    raise FileNotFoundError(
        f"Expected one main Device2 MAT file (excluding b/check); found {len(device2_candidates)}."
    )

device2_path = device2_candidates[0]
device2_contents = loadmat(device2_path, squeeze_me=True, struct_as_record=False)
if "measurement" not in device2_contents:
    raise KeyError(f"No 'measurement' variable in {device2_path.name}")
transient_records = np.atleast_1d(device2_contents["measurement"].transient)
print(f"File: {device2_path.name}; transient captures: {len(transient_records)}")
print("First transient time-domain fields:", transient_records[0].timeDomain._fieldnames)

record_index = 1
if record_index >= len(transient_records):
    raise IndexError(f"Capture {record_index} is unavailable in {device2_path.name}")
record = transient_records[record_index]
time_domain = record.timeDomain
if "dt" not in time_domain._fieldnames:
    raise KeyError("This transient record has no 'dt' sample-interval field.")
channel_name = "collectorEmitterVoltage"
if channel_name not in time_domain._fieldnames:
    raise KeyError(f"This transient record has no {channel_name!r} field.")

dt_seconds = float(np.asarray(time_domain.dt).item())
waveform = np.asarray(getattr(time_domain, channel_name), dtype=float).squeeze()
if waveform.ndim != 1 or waveform.size < 2 or not np.isfinite(waveform).all():
    raise ValueError(f"Expected a finite 1D waveform; got shape={waveform.shape}")
if not np.isfinite(dt_seconds) or dt_seconds <= 0:
    raise ValueError(f"Invalid within-record sample interval: {dt_seconds}")

sample_rate_hz = 1 / dt_seconds
nyquist_hz = sample_rate_hz / 2
time_seconds = np.arange(waveform.size) * dt_seconds
print(f"Capture date from file: {record.date}")
print(f"Samples: {waveform.size:,}")
print(f"dt: {dt_seconds * 1e9:.1f} ns; sample rate: {sample_rate_hz / 1e6:.2f} MHz")
print(f"Nyquist frequency: {nyquist_hz / 1e6:.2f} MHz")
print(f"Capture duration: {time_seconds[-1] * 1e3:.3f} ms")

### Plot the capture before processing

The left panel shows the complete short capture, while the right panel zooms in on one observed switching edge. The edge location is chosen for illustration after inspecting this actual capture; different records can have different timings and sample intervals. This is a high-speed event waveform, not a continuous aging trend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
axes[0].plot(time_seconds * 1e3, waveform, linewidth=0.7)
axes[0].set(title="Full transient capture", xlabel="Capture-relative time (ms)", ylabel="Recorded value (unit to verify)")
axes[0].grid(alpha=0.25)

edge_start_us, edge_end_us = 570, 610
edge_window = (time_seconds * 1e6 >= edge_start_us) & (time_seconds * 1e6 <= edge_end_us)
if not edge_window.any():
    raise ValueError("Chosen example edge window contains no samples; inspect the full capture first.")
axes[1].plot(time_seconds[edge_window] * 1e6, waveform[edge_window], linewidth=0.9)
axes[1].set(title=f"Switching-edge view ({edge_start_us}–{edge_end_us} µs)", xlabel="Capture-relative time (µs)", ylabel="Recorded value (unit to verify)")
axes[1].grid(alpha=0.25)

sampling_figure = project_root / "results" / "figures" / "signal_processing_transient_sampling.png"
sampling_figure.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(sampling_figure, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {sampling_figure.relative_to(project_root)}")

### Sampling: what the two time bases mean

- **Steady-state records:** typically one record about every 0.2 seconds, with occasional gaps. This cadence can follow slow thermal/electrical behavior, but cannot resolve nanosecond switching edges.
- **Transient captures:** individual, 125,000-sample waveform records with a sample interval stored per record. For this dataset, the interval differs between records (observed values include 4, 8, and 16 ns). At 16 ns, for example, the sample rate is 62.5 MHz and Nyquist is 31.25 MHz; at 8 ns they are 125 MHz and 62.5 MHz, respectively.
- **Capture time versus aging time:** the waveform x-axis spans only a fraction of a millisecond or a few milliseconds. The `date`/`timeEpoch` attached to a transient says when that capture was recorded during the experiment. Do not join captures and pretend the sparse gaps between them were sampled.

The dataset paper describes oscilloscope hardware, but do not substitute a hardware specification for each file's actual `dt`. If `dt` changes, sample rate, Nyquist frequency, frequency axes, and filter design change with it.

## 3. Noise and measurement quality

A jagged or ringing trace is not necessarily random noise: switching edges, overshoot, and ringing may be physically meaningful. The dataset README explicitly reports approximately 600 mA of collector-current drift, missing transient measurements, and incorrectly scaled steady-state measurements. Drift is a systematic measurement problem; a low-pass filter does not correct calibration or drift, and could hide a real event.

Before suppressing a component as noise, ask whether it is repeatable, how the probe/acquisition chain behaves, whether it aligns with a switching event, and what measurement band is physically relevant. This data alone does not let us label every high-frequency fluctuation as noise.

## 4. Filtering: demonstrate the trade-off, not a recommended setting

A low-pass Butterworth filter can suppress frequency content above its cutoff. SciPy's `butter(..., fs=..., output="sos")` builds numerically stable second-order sections, and `sosfiltfilt` filters forward and backward to avoid phase delay. The cutoff below is deliberately an **arbitrary classroom illustration**, not a validated noise cutoff or a processing choice for this experiment. Compare its effect on the switching edge: filtering can round off or attenuate the very fast transition we may want to characterize.

We filter one capture as a whole so the edge is not an artificial boundary of the extracted zoom. We do not filter across different captures, across timestamp gaps, or across different sample intervals.

In [ ]:
illustration_cutoff_hz = 5e6
if not 0 < illustration_cutoff_hz < nyquist_hz:
    raise ValueError("The illustrative cutoff must be below this capture's Nyquist frequency.")

sos = signal.butter(
    N=4,
    Wn=illustration_cutoff_hz,
    btype="lowpass",
    fs=sample_rate_hz,
    output="sos",
)
filtered_waveform = signal.sosfiltfilt(sos, waveform)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(time_seconds[edge_window] * 1e6, waveform[edge_window], label="Original capture", linewidth=0.9)
ax.plot(time_seconds[edge_window] * 1e6, filtered_waveform[edge_window], label=f"Illustrative {illustration_cutoff_hz / 1e6:g} MHz low-pass", linewidth=1.2)
ax.set(title="Illustrative filter changes the observed switching edge", xlabel="Capture-relative time (µs)", ylabel="Recorded value (unit to verify)")
ax.grid(alpha=0.25)
ax.legend()
filter_figure = project_root / "results" / "figures" / "signal_processing_filter_tradeoff.png"
fig.savefig(filter_figure, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {filter_figure.relative_to(project_root)}")
print("The filtered waveform exists only in memory; no filtered signal was saved or substituted for raw data.")

### Filtering takeaway

The plot shows the consequence of the chosen cutoff, not proof that the original trace needed filtering. Choose a filter only after defining the signal component to preserve and the interference to remove, using actual sample intervals and measurement-system knowledge. Do not low-pass the 5-records/second steady-state series with this transient's cutoff, and do not filter across its irregular timestamp gaps as if it were uniformly sampled. Do not expect filtering to fix the reported 600 mA current drift or incorrect scale.

## 5. Windowing: define a local event interval

Windowing selects a time region for analysis. It can help compare the pre-switch, switching, and post-switch parts of a transient. The intervals below only illustrate segmentation of this one observed trace; their widths are not validated PHM settings. A fair later comparison would align equivalent switching events and verify operating conditions across captures before measuring any summary statistic.

In [ ]:
window_edges_us = [560, 580, 600, 620]
time_us = time_seconds * 1e6
window_masks = [
    (time_us >= start) & (time_us < end)
    for start, end in zip(window_edges_us[:-1], window_edges_us[1:])
]

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(time_us[edge_window], waveform[edge_window], color="black", linewidth=0.9)
colors = ["#8ecae6", "#ffb703", "#90be6d"]
names = ["pre-switch window", "edge window", "post-switch window"]
for mask, start, end, color, label in zip(
    window_masks, window_edges_us[:-1], window_edges_us[1:], colors, names
):
    count = int(mask.sum())
    ax.axvspan(start, end, color=color, alpha=0.25, label=f"{label}: {count} samples")
ax.set_xlim(edge_start_us, edge_end_us)
ax.set(title="Illustrative time windows around one observed switching edge", xlabel="Capture-relative time (µs)", ylabel="Recorded value (unit to verify)")
ax.grid(alpha=0.25)
ax.legend()
window_figure = project_root / "results" / "figures" / "signal_processing_event_windows.png"
fig.savefig(window_figure, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {window_figure.relative_to(project_root)}")

These short event windows are a way of organizing a waveform, not evidence that its windows correspond to aging stages. For slow steady-state records, time windows also need care: select them from real timestamps, preserve file/device identity, and check acquisition gaps. A fixed number of rows does not necessarily represent a fixed time interval when there are missing records.

## 6. Normalization: preserve meaning before scaling

Common transformations include **centering** (`x - mean`) and **z-score standardization** (`(x - mean) / standard deviation`). They change the numerical scale but not the source measurement. If we standardize each transient separately, however, we remove its mean and scale by its own variability; this can erase between-capture amplitude differences that might matter in PHM. Scaling records by their own minimum/maximum can erase even more.

This small calculation makes the transformation explicit, but we do not overwrite, plot as physical values, or save the normalized signal. The raw transient `waveform` remains unchanged.

In [ ]:
waveform_mean = waveform.mean()
waveform_std = waveform.std()
if waveform_std == 0:
    raise ValueError("Cannot standardize a constant waveform.")

waveform_zscore = (waveform - waveform_mean) / waveform_std
print(f"Original: mean={waveform_mean:.6g}, standard deviation={waveform_std:.6g}")
print(f"Temporary z-score copy: mean={waveform_zscore.mean():.6g}, standard deviation={waveform_zscore.std():.6g}")
print("Use the z-score only to compare standardized shape; do not interpret it as volts or a degradation indicator.")

In a later predictive model, fit a scaler on training data only and apply those learned parameters to validation/test data; fitting it on all devices leaks information about held-out data. For physical interpretation or candidate health indicators, retain and compare calibrated engineering units whenever possible. We are not fitting a model or scaler in this phase.

## 7. FFT and frequency-domain analysis: when is it justified?

An FFT decomposes a uniformly sampled signal into sinusoidal frequency components. It is most interpretable for a sufficiently long, uniformly sampled signal that is approximately stationary over the analysis window, or for a carefully isolated periodic/ringing component. The transient shown here is a single switching event, not a steady periodic recording. Its spectrum would mix the edge shape, capture duration, chosen window, possible ringing, and measurement-chain response. The within-record sample rate also varies.

The MATLAB records contain a `frequencyDomain` structure, so inspect the **actual** array sizes before assuming spectra are populated. For the inspected Device 2 capture, frequency amplitude/phase arrays are empty. Do not fabricate a frequency-domain feature from those empty arrays.

In [ ]:
frequency_domain = record.frequencyDomain
frequency_fields = getattr(frequency_domain, "_fieldnames", [])
frequency_shapes = {
    name: np.asarray(getattr(frequency_domain, name)).shape
    for name in frequency_fields
}
frequency_shapes

**Decision for this dataset and phase:** do not calculate or save FFT-based degradation features yet. An FFT of this isolated edge can be demonstrated later if we formulate a specific switching/ringing question, verify the sample-time metadata, choose an appropriate event window, and account for measurement bandwidth and acquisition settings. A spectral peak alone is not a degradation indicator. The saved figures and temporary filter/normalization arrays in this notebook are demonstrations, not model inputs.

## Phase 4 learning check

1. What does a transient record's `dt` tell you? How do you calculate its sample rate and Nyquist frequency?
2. Why must separate transient captures be processed separately, especially when their `dt` differs?
3. In the filter comparison, what part of the switching event changed? Why might smoothing be harmful here?
4. Why can z-score normalization make two waveforms look more alike even if their physical amplitudes differ?
5. Why is an FFT of one switching edge not automatically a frequency-domain degradation feature? What evidence would make spectral analysis more defensible?
6. Which documented current/scale/transient limitations affect your interpretation of the figures?

Run the cells in order in VS Code and share your answers and any unexpected results. The next phase explores candidate features from the actual transient captures; do not treat them as validated Health Indicators or RUL inputs.